In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
import gzip
from pathlib import Path


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl
import os
from pathlib import Path

In [ ]:
from pathlib import Path
import os
import pandas as pd


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve() if sig_env else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ------------------------------------------------------------------
# MAIN (NO SORTING)
# ------------------------------------------------------------------

FINAL_PATH, OUT_DIR = _resolve_orion_paths("arbride")

print("Using FINAL parquet (no sorting):", FINAL_PATH)


In [ ]:
from __future__ import annotations

# -- ratings, alpha, gamma, delta, lambda from the all-history EVENT table the streaming pass produces ------------
# THIS CELL IS ARB•FADE'S OWN, UNCHANGED - not one number in it differs (operator, 2026-11-26: "обчислення
# рейтингів і класів точно такі ж самі"). The thesis difference lives entirely in the cell below, which
# decides what `status` and `pnl` mean before this one ever sees them:
#   * OUTCOME RULE: a trade is a WIN when the deviation EXTENDED - the ride. ARB•FADE scores the
#     opposite path, normalisation, with the identical vocabulary; both are binary since 2026-11-26.
#     mean_pnl / sum_pnl / pnl_lb / gamma run on the real P&L (pnl_pct) and are orientation-aware for
#     free: they read THIS file's pnl column, so gamma finds the level at which RIDING paid.
#   * CLASS LIST: three overnight classes x two event models = six keys on the wire
#     (PRE_ACT / PRE_PAS / BLUE_ACT / BLUE_PAS / ARK_ACT / ARK_PAS; "cls" UPPERCASE, "sign" pos/neg,
#     the OPG wire convention). The UI shows them as two toggles, never six buttons. The windows and
#     the two models live in the cell below; nothing in THIS cell knows a class name except the order.
# "pos"/"neg" still label the DIRECTION OF THE DEVIATION, never the trade taken - so a "pos" row here is a
# LONG (buy the leg that ran ahead), where in ARB•FADE it is a SHORT. Same labelling as every other family.
import math

import numpy as np
import pandas as pd

ARB_CLASSES = ("PRE_ACT", "PRE_PAS", "BLUE_ACT", "BLUE_PAS", "ARK_ACT", "ARK_PAS")
ARB_GAMMA_CFG = {"gamma_floor": 0.60, "gamma_max": 3.0, "gamma_step": 0.1, "gamma_min_n": 10, "gamma_z": 1.96}


def _wilson_lb(k, n, z=1.96):
    k = np.asarray(k, float)
    n = np.asarray(n, float)
    with np.errstate(all="ignore"):
        p = k / n
        den = 1 + z * z / n
        c = p + z * z / (2 * n)
        r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
        return np.where(n > 0, (c - r) / den, np.nan)


def _modal_value(values, step=0.1, min_n=3):
    """The value the deviation takes MOST OFTEN (mode, not RMS): |values| binned at `step`, the modal bin wins,
    reported as the MEAN of the raw readings in it. NaN below min_n readings."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if v.size < min_n:
        return float("nan")
    bins = np.round(v / step) * step
    uniq, counts = np.unique(bins, return_counts=True)
    best = uniq[np.argmax(counts)]
    return float(np.mean(v[np.abs(bins - best) < 1e-9]))


def compute_alpha(events, step=0.1, min_n=10):
    """ALPHA per (ticker, cls, sign): the modal |start deviation| of every birth - the level this ticker's
    divergences usually START at, gate-free (every birth the state machine produced, tradable or not)."""
    if not len(events):
        return pd.DataFrame(columns=["ticker", "cls", "sign", "alpha", "alpha_n"])
    rows = []
    for (tk, cls, sg), g in events.groupby(["ticker", "cls", "sign"], sort=False):
        d = np.abs(g["entry_dev"].to_numpy(dtype=float))
        rows.append((tk, cls, sg, _modal_value(d, step, min_n), int(d.size)))
    return pd.DataFrame(rows, columns=["ticker", "cls", "sign", "alpha", "alpha_n"])


def compute_lambda(events, min_n=2):
    """LAMBDA per ticker: root-mean-square (sqrt(mean(x^2)), not std) of the signed start deviation over every
    birth, all classes together - the ticker's own typical divergence size. NaN below min_n births."""
    if not len(events):
        return pd.Series(dtype=float, name="lambda").rename_axis("ticker")
    ok = events.groupby("ticker")["entry_dev"].transform("count") >= min_n
    sq = events.loc[ok, "entry_dev"].astype(float) ** 2
    return sq.groupby(events.loc[ok, "ticker"]).mean().pow(0.5).rename("lambda").rename_axis("ticker")


def compute_ratings(events):
    """The three numbers the operator asked each cell to carry (2026-11-26), per (ticker, cls, sign):
      win_rate  wins / total - the share of situations that EXTENDED to the ride's target. Binary:
                there is no hard/soft split any more.
      total     how many situations there were at all.
      sum_pnl   the SUM of the real Stack% move over them - how much the ticker actually returned,
                which is a different question from how often it worked.
    Plus mean / median pnl and the 95% lower bound of the mean on the real P&L (pnl_pct)."""
    cols = ["ticker", "cls", "sign", "total", "wins", "win_rate", "win_rate_lb", "losses",
            "sum_pnl", "mean_pnl", "median_pnl", "pnl_lb", "pnl_n"]
    if not len(events):
        return pd.DataFrame(columns=cols)
    e = events.assign(_conv=(events["status"] == "conv").astype(np.int32))
    g = e.groupby(["ticker", "cls", "sign"], sort=False)
    out = g.agg(total=("status", "size"), wins=("_conv", "sum"),
                sum_pnl=("pnl", "sum"),
                mean_pnl=("pnl", "mean"), median_pnl=("pnl", "median"), sd_pnl=("pnl", "std"),
                pnl_n=("pnl", "count")).reset_index()
    out["losses"] = out["total"] - out["wins"]
    out["win_rate"] = out["wins"] / out["total"]
    out["win_rate_lb"] = _wilson_lb(out["wins"], out["total"])
    with np.errstate(all="ignore"):
        se = out["sd_pnl"] / np.sqrt(out["pnl_n"])
        out["pnl_lb"] = np.where(out["pnl_n"] > 1, out["mean_pnl"] - 1.96 * se, np.nan)
    return out[cols]


def compute_gamma(events, cfg):
    """GAMMA per (ticker, cls, sign): the LOWEST level L >= gamma_floor at which births reaching |start dev| >= L
    have a positive 95% lower bound on mean pnl (OPG's own definition)."""
    cols = ["ticker", "cls", "sign", "gamma", "gamma_n", "gamma_mean", "gamma_rate"]
    if not len(events):
        return pd.DataFrame(columns=cols)
    ev = events[events["pnl"].notna()]
    rows = []
    for (tk, cls, sg), g in ev.groupby(["ticker", "cls", "sign"], sort=False):
        dev = np.abs(g["entry_dev"].to_numpy(dtype=float))
        pnl = g["pnl"].to_numpy(dtype=float)
        lvl = float(cfg["gamma_floor"])
        while lvl <= cfg["gamma_max"] + 1e-9:
            m = dev >= lvl - 1e-9
            n = int(m.sum())
            if n < cfg["gamma_min_n"]:
                break
            mp = float(pnl[m].mean())
            se = float(pnl[m].std(ddof=1)) / math.sqrt(n) if n > 1 else math.inf
            if mp - cfg["gamma_z"] * se > 0:
                rows.append((tk, cls, sg, lvl, n, mp, float((pnl[m] > 0).mean())))
                break
            lvl = round(lvl + cfg["gamma_step"], 6)
    return pd.DataFrame(rows, columns=cols)


def compute_delta(events, last_n=5):
    """DELTA per (ticker, sign): median first PRE print - the |start dev| of the FIRST PRE birth of each night,
    over the last `last_n` nights whose first birth had that sign (what the bridge reads as static.delta)."""
    cols = ["ticker", "sign", "delta"]
    # PRE_ACT, not PRE_PAS: delta is "the |start dev| of the night's FIRST print", and ACTIVE's
    # first event of the night IS that print (PASSIVE's single event is the same one). ACTIVE is
    # also the class alpha and gamma are read off, so the levels stay on one population.
    pre = events[(events["cls"] == "PRE_ACT") & (~events["is_alt"])]
    if not len(pre):
        return pd.DataFrame(columns=cols)
    first = (pre.sort_values(["ticker", "date", "start_min"], kind="stable")
                .drop_duplicates(["ticker", "date"], keep="first"))
    rows = []
    for (tk, sg), g in first.groupby(["ticker", "sign"], sort=False):
        d = g.sort_values("date", ascending=False)["entry_dev"].head(last_n).to_numpy(dtype=float)
        rows.append((tk, sg, float(np.median(d))))
    return pd.DataFrame(rows, columns=cols)


def build_summary(events, static_by_ticker, cfg=ARB_GAMMA_CFG, min_events_per_ticker=10):
    """alpha + ratings + gamma (+ delta, lambda, identification) joined into one per-(ticker, cls, sign) table."""
    prim = events[~events["is_alt"]]            # the 04:02 "alt" PRE track is trade-log only, never a rating
    keep = prim.groupby("ticker")["status"].transform("size") >= int(min_events_per_ticker)
    prim = prim[keep]
    ratings = compute_ratings(prim)
    out = (ratings.merge(compute_gamma(prim, cfg), on=["ticker", "cls", "sign"], how="left")
                  .merge(compute_alpha(prim), on=["ticker", "cls", "sign"], how="left"))
    delta = compute_delta(prim)
    out = out.merge(delta, on=["ticker", "sign"], how="left")
    out.loc[~out["cls"].str.startswith("PRE"), "delta"] = np.nan         # delta exists for PRE only
    lam = compute_lambda(prim)
    out["lambda"] = out["ticker"].map(lam) if len(lam) else np.nan
    for _f in ("sigma", "bench", "corr", "beta"):
        out[_f] = [(static_by_ticker.get(t) or {}).get(_f) for t in out["ticker"]]
    order = {c: i for i, c in enumerate(ARB_CLASSES)}
    out["_o"] = out["cls"].map(order)
    return out.sort_values(["ticker", "_o", "sign"]).drop(columns="_o").reset_index(drop=True), lam


In [ ]:
from __future__ import annotations


def devsig_rolling_perf_exporter(
    input_path: str,
    output_path: str = "ARBRIDE/arbride_rolling_perf.json.gz",
    ratings_path: str = "ARBRIDE/arbride_ratings.csv",
    *,
    windows_days=(5, 20, 40, 65),
    dev_thr: float = 0.60,      # start 0.6 sigma (dev_sig is in sigma units); target ext_thr = 1.8
    # THE RIDE TARGET, the mirror of ARB•FADE's norm_thr. FADE wins when |dev_sig| collapses to 0.20,
    # a third of the 0.60 trigger; the ride wins when it EXTENDS to three times it. Same factor, the
    # other way; keep the two in step if dev_thr ever moves, or the pair stops being a mirror and the
    # comparison between them stops meaning anything.
    ext_thr: float = 1.80,
    # NOISE GUARD. An event used to be born the instant ONE tick showed |dev_sig| >= dev_thr, so a
    # single bad print (a stale quote, a one-tick spike that is gone on the next row) opened an event,
    # was scored like a real one and dragged every rate. min_hold_ticks=2 requires
    # that many CONSECUTIVE ticks of this ticker, inside the class's birth window, at or beyond
    # dev_thr with the SAME sign; the event is born on the confirming tick (entry at the price
    # you could actually have acted on - the first tick is only a candidate). A tick below the
    # threshold, of the other sign, outside the window, or a new session day breaks the run. 1 = the
    # old behaviour. Runs that break before confirming are counted and printed as rejected blips.
    min_hold_ticks: int = 2,
    # THE MODEL (user, 2026-09-21): every deviation that appears in a class's birth window is TAKEN - PRE and
    # BLUE from 21:00, ARK from 04:00. What happens next is the VARIANT's job: an ACTIVE event is closed by the
    # resolution itself, a PASSIVE one is held to 09:30 whatever happens. Nothing is cut at 00:00 or 04:00. The ONLY thing
    # thrown away is `ignore_windows`, ALWAYS: rows inside them are dropped before the loop, so no situation can
    # start there, and a dropped tick does not count toward min_hold_ticks. Minutes, both ends inclusive; a window
    # only matters to a class whose own window contains it - which, now that every class is overnight, is all
    # of them: 03:58-04:02 straddles the BLUE/ARK boundary and 07:58-08:02 sits inside ARK.
    ignore_windows=(((3, 58), (4, 2)), ((7, 58), (8, 2))),
    # DISCONTINUITY GUARD (now OFF by default, see below). Measured on the published trade logs (2026-09-20): PairFlux exits pile up
    # at exactly 00:00 (1183 vs 15 at 23:59, and 99.8% of them "win"), Arbitrage births at 00:00 are
    # 14x the minute before (1445 vs 106), and births/exits spike again right after 04:00 (04:06: 996
    # vs ~50 a minute). A market does not do that on the clock - the feed's Stack% baseline / session
    # rolls over there, so a "deviation" appears or vanishes mechanically. No event may be BORN in the
    # disc_guard_min minutes after any of these clock times, and a run of ticks never spans one.
    # (Events that are ALIVE across a boundary are only flagged - `spans_disc` - not dropped.)
    # OFF by default (2026-09-21): an empty `discontinuities` guards nothing and flags nothing (`spans_disc` is
    # then always False). Set e.g. ((0, 0), (4, 0)) with disc_guard_min=10 to bring the old birth guard back.
    discontinuities=(),
    disc_guard_min: int = 0,
    # bucket the START (birth) deviation - no lookahead, this is what you'd actually
    # know the instant you'd enter. dev_thr is the floor (nothing is born below it).
    start_bin_step: float = 0.2,
    start_bin_max: float = 3.0,
    min_events_per_ticker: int = 10,
    min_events_per_cell: int = 2,
    # $ position size assumed for every trade when turning an event into P&L.
    position_usd: float = 1000.0,
    # emit the raw per-event trade log (needed to draw a P&L curve for an arbitrary,
    # user-chosen "start" threshold - the bucket grid below is fixed-step and can't
    # answer "what if start=0.47" on its own).
    include_trade_log: bool = True,
    BENCH_NUM_FIELD: str = "Bench%",
    STOCK_NUM_FIELD: str = "Stack%",
    parquet_batch_log_every: int = 10,
):
    """
    Rolling recency performance, split by class x sign x start-deviation bucket x
    trailing window (default last 5/20/40/65 sessions), PLUS a raw per-event trade log
    for P&L-curve rendering. Built for the "Scout" UI: pick a ticker/class/sign/
    window, set a "start" threshold (in %, sigma, gamma- or delta-relative units -
    gamma/delta come from the companion best_params.jsonl.gz, joined by ticker),
    and get both a hit-rate and a $ P&L curve, entirely client-side against this
    one file (no backend recompute per interaction).

    WHY START, NOT PEAK: the existing v13 onefile buckets by the PEAK deviation,
    which is only known after the fact. Bucketing by the peak here would let a
    trade that started at 0.31 and later widened to 1.40 count as a "1.2-1.4"
    signal, which you could not have known at entry. Both the bucket grid AND the
    trade log key off the deviation observed the moment the class fires.

    WHY WINDOWS ARE FILE-RELATIVE: final.parquet is a static historical file, not a
    live feed, so "last 5/20/40/65 days" means the 5/20/40/65 most recent SESSION DATES
    that are actually present in the file (per the `date` column), not
    datetime.date.today(). A session's rank is 0 for the most recent date, 1 for
    the one before it, etc.; an event's start_date qualifies for window W iff its
    rank < W. Windows are cumulative (last-5 subset-of last-20 subset-of last-40).
    The trade log carries every event within the LARGEST window (default 40); the
    frontend slices it down to 5/20 using meta.recent_dates (see below) rather than
    the backend emitting three overlapping copies of the same trades.

    Three overnight classes, each in two VARIANTS that are different event models (the
    operator's spec, 2026-11-26). Six keys on the wire, two toggles on screen.
      PRE    births 21:00(prev day)-09:00, deadline 09:30   (wraps midnight)
      BLUE   births 21:00(prev day)-04:00, deadline 04:00   (wraps midnight)
      ARK    births 04:00-09:00,           deadline 09:00
      ACTIVE   the event ENDS on the resolution, P&L marked there, and the track re-arms -
               one night can produce several events.
      PASSIVE  one entry per session, out at 09:30 for every class, P&L marked there.
    A situation RESOLVES when |dev_sig| >= ext_thr - and that is the whole outcome now; the hard/soft split is gone.

    P&L MODEL (real Stack% move, not a synthetic sigma-compression number):
      - entry price marker = Stack% at the tick the class was born (start_stock).
      - exit price marker  = Stack% at the tick the trade actually came off: the
        RESOLUTION for a resolved ACTIVE event, the window close otherwise, and always
        09:30 for PASSIVE. ACTIVE and PASSIVE are therefore not comparable trade for
        trade on purpose - they are two different ways of holding the same situation,
        which is the whole point of having both.
      - direction: a "pos" event (dev_sig>0: the stack ran ahead of the benchmark)
        is a LONG; a "neg" event is a SHORT. This is the CONTINUATION trade - you
        ride the leg that got ahead instead of fading it, which is the one thing
        this file does differently from ARB•FADE.
      - pnl_pct = (exit_stock - start_stock) for pos [long], (start_stock -
        exit_stock) for neg [short] - both in percentage POINTS. Exactly ARB•FADE's
        pnl with the sign flipped, so on any shared event the two sum to zero
        before costs.
      - pnl_usd = position_usd * pnl_pct / 100.

    OUTPUT: a single gzip-if-.gz-suffixed JSON file:
      {"meta": {..., "recent_dates": [...], "position_usd": 1000.0},
       "rows": [ {ticker, bench, corr, beta, sigma,
                   classes: {pre: {pos: {"0.30-0.50": {"w5": {...}, ...}}}}},
                   trades: [{cls, sign, date, start_dev, start_gap_pct, status, pnl_pct, pnl_usd,
                             start_minute_idx, peak_minute_idx, peak_dev_abs, end_dev_abs,
                             end_gap_pct}, ...] (chronological) },
                  ... ]}
    start_minute_idx/peak_minute_idx are NY minute-of-day at birth/peak, wrapped for the four
    tracks that can start in the evening (PRE and BLUE, in both variants): a prior-day 21:00-23:59
    birth is negative, h*60+m-1440, so PRE's axis runs -180..570 and BLUE_ACT's -180..240 instead of
    a 21:00-23:59 tail stranded past their own close - the same convention TradingTool's
    sessionTimeChartRange already uses. ARK starts at 04:00 and stays on the plain axis (240..570). peak_dev_abs/end_dev_abs are |dev_sig| at the peak tick
    and at the window-close mark, respectively - both None if the event never ticked (should not
    happen; a born event always ticks at least once, at birth). end_gap_pct is start_gap_pct's
    companion - Stack%-Bench% at the window-close mark instead of at birth - kept as its OWN
    field rather than derived: pnl_pct is built from start_stock/last_stock alone (see
    _pnl_pct), so end_gap_pct - start_gap_pct is NOT pnl_pct whenever the bench itself moved
    between birth and close (the normal case).
    Sparse: a (class, sign, bucket, window) cell is only present if it has at least
    one event; a cell's rate/avg_pnl fields are null (raw counts kept) below
    min_events_per_cell. `start_gap_pct` on a trade is the raw (unnormalized)
    Stack%-Bench% gap at entry - the "%" unit mode; `start_dev` is the sigma-
    normalized dev_sig - the "sigma" unit mode. Gamma/delta unit modes are ratios
    against that ticker's own static gamma/delta level from best_params.jsonl.gz.
    """
    import gc, gzip, json, math, time
    from collections import Counter, defaultdict
    from datetime import datetime, timedelta
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    CLASSES = ("pre_act", "pre_pas", "blue_act", "blue_pas", "ark_act", "ark_pas")
    # Every track is a rated class now - the old trade-log-only "pre_alt" track is gone, because
    # ARK (births 04:00-09:00) is exactly the model it was standing in for, and it is rated.
    TRACKS = CLASSES
    SIGNS = ("pos", "neg")
    WINDOWS = tuple(sorted(windows_days))
    MAX_WINDOW = WINDOWS[-1]

    def _open_text(path: str, mode: str = "wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    _math_isfinite = math.isfinite

    def is_finite_num(x) -> bool:
        # 7.2M calls per two row groups, and after the .tolist() conversions above almost every one
        # of them is a plain Python float. math.isfinite is exactly np.isfinite(float(x)) for that
        # type - same answer for NaN, +-inf and every finite value - and skips the float() call,
        # the numpy dispatch and the try/except frame. Anything else still takes the old path.
        if type(x) is float:
            return _math_isfinite(x)
        try:
            return np.isfinite(float(x))
        except Exception:
            return False

    def in_range(t, a, b):
        return (t is not None) and (a <= t <= b)

    def in_range_wrap(t, a, b):
        if t is None:
            return False
        if a <= b:
            return a <= t <= b
        return t >= a or t <= b

    # ---------------- classes / windows / the two event models ----------------
    # 2026-11-26, the operator's own spec. THREE overnight classes, each in TWO variants that are
    # different EVENT MODELS - not two readings of one set of events. Entries only:
    #
    #   PRE   21:00 -> 09:00      BLUE  21:00 -> 03:59      ARK  04:00 -> 09:00
    #
    #   ACTIVE   open on the trigger, run to the CLOSING bar, close THERE with the P&L marked at
    #            that tick - and then, if the ticker signals again, open another and watch again.
    #            Several a night. Whatever is still open at 09:30 is force-closed there and counts
    #            as a failure. A position you work.
    #   PASSIVE  ONE entry per session - the first of the window - and out at 09:30 ALWAYS,
    #            whatever the entry time and whatever happened in between. A position you do not
    #            touch, and the question asked of it is a different one: see finalize_event.
    #
    # Six keys on the wire, two toggles on screen. 09:30 is the one formal close for everything; a
    # class bounds only when a situation may be TAKEN.
    PRE_FROM, PRE_TO, PRE_START_TO = (21, 0), (9, 30), (9, 0)
    NIGHT_SPLIT = (4, 0)        # where BLUE ends and ARK begins - the Caesar bar's own boundary
    BLUE_START_TO = (3, 59)     # BLUE takes entries up to 03:59, the operator's own bound

    # A CLASS IS A BIRTH WINDOW, NOT AN EXIT (operator, 2026-11-26). Every track runs to 09:30; what
    # the class decides is when a situation may be TAKEN:
    #   PRE   21:00 -> 09:00      BLUE  21:00 -> 03:59      ARK  04:00 -> 09:00
    # so PRE is exactly BLUE + ARK as a population of entries, and the two halves are the question
    # "does it matter WHEN in the night it started?".
    #
    # birth window, class window (the span an event may live in), wraps midnight, is passive
    TRACK_SPEC = {
        "pre_act":  ((PRE_FROM, PRE_START_TO), (PRE_FROM, PRE_TO), True, False),
        "pre_pas":  ((PRE_FROM, PRE_START_TO), (PRE_FROM, PRE_TO), True, True),
        "blue_act": ((PRE_FROM, BLUE_START_TO), (PRE_FROM, PRE_TO), True, False),
        "blue_pas": ((PRE_FROM, BLUE_START_TO), (PRE_FROM, PRE_TO), True, True),
        "ark_act":  ((NIGHT_SPLIT, PRE_START_TO), (NIGHT_SPLIT, PRE_TO), False, False),
        "ark_pas":  ((NIGHT_SPLIT, PRE_START_TO), (NIGHT_SPLIT, PRE_TO), False, True),
    }
    BIRTH_WINDOWS = {k: v[0] for k, v in TRACK_SPEC.items()}
    CLASS_WINDOWS = {k: v[1] for k, v in TRACK_SPEC.items()}
    WRAP = {k: v[2] for k, v in TRACK_SPEC.items()}
    IS_PASSIVE = {k: v[3] for k, v in TRACK_SPEC.items()}

    # The `ignore_windows` rows (03:58-04:02, 07:58-08:02) are dropped before the loop, so in
    # practice BLUE's 04:00 mark is the last print at or before 03:57 and ARK's first possible birth
    # tick is 04:03.

    def _plus_grace(t, mins=10):
        v = t[0] * 60 + t[1] + mins
        return ((v // 60) % 24, v % 60)

    # An event is FINALISED only once the clock is safely past its own window, because a wrapping
    # class cannot tell "still inside my window" from "long past it" by wall time alone (03:00 is
    # both, for PRE). Ten minutes past the close, resolved as a real datetime off the event's own
    # start_dt - exactly what v13 did for PRE with its hardcoded 09:40, now one bound per track.
    GRACE_TO = {c: _plus_grace(w[1]) for c, w in CLASS_WINDOWS.items()}

    IGNORE_WINDOWS = [((int(a[0]), int(a[1])), (int(b[0]), int(b[1]))) for a, b in ignore_windows]

    def minute_of_day(dt, cls: str) -> "int | None":
        if not isinstance(dt, datetime):
            return None
        total = dt.hour * 60 + dt.minute
        # The wrapping tracks (PRE and BLUE, both variants) only ever tick in [21:00,23:59]
        # (evening) or [00:00,09:40] (morning) - noon is a safe splitter between the two. ARK never
        # sees the evening at all, so it stays on the plain 0..1439 axis.
        return total - 1440 if (WRAP.get(cls) and total >= 720) else total

    def start_bin_label(abs_dev: float) -> str:
        if abs_dev >= start_bin_max:
            return f"{start_bin_max:.1f}+"
        v = max(float(dev_thr), float(abs_dev))
        idx = int((v - dev_thr) / start_bin_step + 1e-9)
        lo = dev_thr + idx * start_bin_step
        hi = lo + start_bin_step
        return f"{lo:.2f}-{hi:.2f}"

    # ---------------- pass 1: session-date ranking (file-relative windows) ----------------
    pf = pq.ParquetFile(input_path)
    date_col = "date" if "date" in pf.schema.names else None
    if date_col is None:
        raise KeyError("Input must contain a 'date' column")

    # The distinct dates, in Arrow. The shipped version materialised the whole column into pandas
    # and then ran astype(str).unique() over 2M values per row group just to learn ~800 dates;
    # pc.unique does it on the Arrow array and hands back only the distinct ones. Verified to
    # produce the identical set. (Row-group statistics cannot replace this: the file is sorted by
    # TICKER, so every group spans the entire date range and min/max tells us nothing about which
    # dates are actually present.)
    import pyarrow.compute as _pc
    seen_dates = set()
    for rg in range(pf.num_row_groups):
        _u = _pc.unique(pf.read_row_group(rg, columns=["date"]).column("date"))
        seen_dates.update(str(x) for x in _u.to_pylist() if x is not None)

    all_dates_sorted = sorted(seen_dates)  # ascending; ISO date strings sort correctly
    rank_from_end = {d: i for i, d in enumerate(reversed(all_dates_sorted))}  # 0 = most recent
    n_sessions = len(all_dates_sorted)
    recent_dates = all_dates_sorted[-MAX_WINDOW:]

    def windows_for_date(date_str: str):
        r = rank_from_end.get(str(date_str))
        if r is None:
            return ()
        return tuple(w for w in WINDOWS if r < w)

    print(f"sessions found: {n_sessions} (most recent: {all_dates_sorted[-1] if all_dates_sorted else None}), "
          f"windows={WINDOWS}")

    # ---------------- per-ticker aggregation state ----------------
    # agg[ticker][cls][sign][bucket][f"w{W}"] -> counts + dev_sum/n + pnl_sum/n
    def new_agg_ticker():
        return {c: {sk: defaultdict(lambda: defaultdict(
                        lambda: {"c": Counter(), "dev_sum": 0.0, "dev_n": 0, "pnl_sum": 0.0, "pnl_n": 0}))
                     for sk in SIGNS} for c in CLASSES}

    agg = {}
    static = {}   # ticker -> {"bench":..., "corr":..., "beta":..., "sigma":...}
    all_events = []   # EVERY event of the whole file (all sessions): source of the all-history ratings
    EV_COLS = ("ticker", "cls", "sign", "date", "start_dev", "start_gap_pct", "status", "pnl_pct",
               "start_minute_idx", "peak_minute_idx", "peak_dev_abs", "end_dev_abs", "end_gap_pct",
               "exit_minute_idx", "birth_gap_min", "spans_disc", "is_alt")

    def _spans(start_min, exit_min):
        # alive across a rollover: the entry baseline and the exit baseline may not be the same one
        if start_min is None or exit_min is None:
            return False
        return any(start_min < d <= exit_min for d in DISC_IDX)

    def record_event(ticker, cls, sign_key, start_date, start_dev, start_gap_pct, status, pnl_pct,
                      start_minute_idx=None, peak_minute_idx=None, peak_dev_abs=None, end_dev_abs=None,
                      end_gap_pct=None, exit_minute_idx=None, birth_gap_min=None, spans_disc=False):
        # One flat tuple per event, whatever its date: the vectorised tail below turns the whole table into
        # ratings / alpha / gamma / delta / lambda, and cuts the trade log to the recent sessions itself.
        # `is_alt` stays in the shape as a constant False: the column is part of the published
        # trade log and the Scout's "FROM 04:02" toggle reads it, so the page disables that toggle
        # by itself instead of breaking on a field that vanished.
        all_events.append((ticker, cls, sign_key, start_date, float(start_dev),
                           start_gap_pct, status, pnl_pct, start_minute_idx, peak_minute_idx, peak_dev_abs,
                           end_dev_abs, end_gap_pct, exit_minute_idx, birth_gap_min, bool(spans_disc),
                           False))

    # ---------------- event state (one live event per track; ACTIVE re-arms, PASSIVE does not) -----
    def _default_event_state():
        return {
            "active": False, "start_dt": None, "start_date": None, "start_dev": np.nan, "start_sign": 0,
            "start_stock": np.nan, "start_bench": np.nan,
            "last_stock": np.nan, "last_bench": np.nan, "last_dev_abs": np.nan,
            "last_dt": None, "birth_gap_min": None,
            # The adverse extreme, kept for the trade log's own charts only - it decides nothing now
            # that the outcome is binary (the old `soft` rule was the one thing that read it).
            "peak_abs": float("inf"), "peak_dt": None,
            "conv_dt": None,
            # ACTIVE re-arm gate: an ended event leaves the track BLOCKED until |dev_sig| falls back
            # under dev_thr, so the next event is a genuinely new divergence rather than the tail of
            # the one that just resolved. See the note where it is cleared.
            "rearm_ok": True,
        }

    # PASSIVE fires once per SESSION, and a session is named after the morning its window ends on -
    # an evening tick belongs to the next calendar day's session. Keyed this way rather than reset on
    # a boundary, because the day roll happens at midnight, in the MIDDLE of PRE's and BLUE's window.
    passive_fired = {}

    def session_key(dt_now, t):
        return (dt_now.date() + timedelta(days=1)).isoformat() if t >= PRE_FROM else dt_now.date().isoformat()

    events_by_ticker_cls = {}  # (ticker, cls) -> event state, only for the CURRENT ticker being streamed
    MIN_HOLD = max(1, int(min_hold_ticks))
    _disc_minutes = set()
    for _dh, _dm in discontinuities:
        for _k in range(int(disc_guard_min)):
            _disc_minutes.add(((_dh * 60 + _dm + _k) % 1440))
    def in_disc_guard(t):
        return (t[0] * 60 + t[1]) in _disc_minutes
    DISC_IDX = [(_dh * 60 + _dm) for _dh, _dm in discontinuities]   # on the PRE-wrapped axis (evening < 0)
    runs = {}                  # (ticker, cls) -> [consecutive qualifying ticks, sign]

    def _end_run(ticker, cls):
        # 13.9M calls per two row groups, and `runs` is empty for almost all of them: a signal has
        # to be mid-run for an entry to exist. Checking that first skips building the (ticker, cls)
        # tuple and the pop - popping from an empty dict does nothing anyway.
        if not runs:
            return
        runs.pop((ticker, cls), None)

    def get_event(ticker, cls):
        key = (ticker, cls)
        e = events_by_ticker_cls.get(key)
        if e is None:
            e = _default_event_state()
            events_by_ticker_cls[key] = e
        return e

    def start_event(ticker, cls, dt_now, ds, dev_now, stock_pct, bench_pct, birth_gap_min=None):
        e = _default_event_state()
        e["active"] = True
        e["birth_gap_min"] = birth_gap_min
        e["last_dt"] = dt_now
        e["start_dt"] = dt_now
        e["start_date"] = ds
        e["start_dev"] = float(dev_now)
        e["start_sign"] = 1 if float(dev_now) >= 0 else -1
        e["start_stock"] = stock_pct
        e["start_bench"] = bench_pct
        e["last_stock"] = stock_pct
        e["trough_abs"] = abs(float(dev_now))
        e["trough_dt"] = dt_now
        events_by_ticker_cls[(ticker, cls)] = e

    def process_tick(ticker, cls, dt_now, dev_now, stock_pct, bench_pct, t, _e=None):
        # The caller already holds the live event dict; looking it up a second time was ~3.5M
        # redundant lookups per two row groups. _e defaults to None so any other call site keeps
        # working unchanged.
        e = _e if _e is not None else get_event(ticker, cls)
        cur_abs = abs(float(dev_now))
        wf, wt = CLASS_WINDOWS[cls]
        inside = in_range_wrap(t, wf, wt) if WRAP[cls] else in_range(t, wf, wt)
        if not inside:
            return
        if is_finite_num(stock_pct):
            e["last_dt"] = dt_now  # WHEN that mark was actually printed: thin names stop printing
            # hours before the window closes, so "exit at 09:30" is really "exit at the last print"
            e["last_stock"] = stock_pct
        if is_finite_num(bench_pct):
            e["last_bench"] = bench_pct  # pairs with last_stock for end_gap_pct
        e["last_dev_abs"] = cur_abs
        if cur_abs < float(e["peak_abs"]):
            e["peak_abs"] = float(cur_abs)
            e["peak_dt"] = dt_now
        if e["conv_dt"] is None and cur_abs >= ext_thr:
            e["conv_dt"] = dt_now
            # ACTIVE exits HERE, on the resolution itself - last_stock was just set to this tick, so
            # the P&L is marked where the trade would really have come off. The track is left idle
            # and may open a new event later in the same window. PASSIVE only REMEMBERS that it
            # resolved; it is still in the position and still exits at 09:30.
            if not IS_PASSIVE[cls]:
                finalize_event(ticker, cls)

    def _pnl_pct(sign_key, start_stock, exit_stock):
        if not (is_finite_num(start_stock) and is_finite_num(exit_stock)):
            return None
        diff = float(exit_stock) - float(start_stock)
        return diff if sign_key == "pos" else -diff  # LONG on pos, SHORT on neg (the ride)

    def finalize_event(ticker, cls):
        e = get_event(ticker, cls)
        if not e["active"]:
            return
        # Binary since 2026-11-26: it worked or it did not. The old hard/soft split is gone - for
        # ACTIVE the resolution IS the exit, so a "partial retrace" has nothing left to mean.
        #
        # The two variants are asked DIFFERENT QUESTIONS, and that is the point of having both:
        #   ACTIVE   did it reach the closing bar at all before 09:30? (the hard rule)
        #   PASSIVE  you never acted, so "did it touch" is not the question - was the deviation
        #            BIGGER at 09:30 than it was at entry?
        # A PASSIVE event whose close never printed has no answer and is scored a failure rather
        # than dropped: it IS a position that sat to 09:30 and ended somewhere.
        if IS_PASSIVE[cls]:
            _d0 = abs(float(e["start_dev"]))
            _d1 = e["last_dev_abs"]
            status = "conv" if (is_finite_num(_d1) and float(_d1) > _d0) else "none"
        else:
            status = "conv" if e["conv_dt"] is not None else "none"
        sk = "pos" if e["start_sign"] > 0 else "neg"

        # The exit mark is whatever last_stock holds: the resolution tick for a resolved ACTIVE
        # event, the window close otherwise (and always the window close for PASSIVE).
        pnl_pct = _pnl_pct(sk, e["start_stock"], e["last_stock"])
        start_gap_pct = None
        if is_finite_num(e["start_stock"]) and is_finite_num(e["start_bench"]):
            start_gap_pct = float(e["start_stock"]) - float(e["start_bench"])
        end_gap_pct = None
        if is_finite_num(e["last_stock"]) and is_finite_num(e["last_bench"]):
            end_gap_pct = float(e["last_stock"]) - float(e["last_bench"])

        record_event(
            ticker, cls, sk, e["start_date"], e["start_dev"], start_gap_pct, status, pnl_pct,
            start_minute_idx=minute_of_day(e["start_dt"], cls),
            peak_minute_idx=minute_of_day(e["peak_dt"], cls),
            peak_dev_abs=e["peak_abs"] if math.isfinite(e["peak_abs"]) else None,
            end_dev_abs=e["last_dev_abs"] if is_finite_num(e["last_dev_abs"]) else None,
            end_gap_pct=end_gap_pct,
            exit_minute_idx=minute_of_day(e["last_dt"], cls),
            birth_gap_min=e["birth_gap_min"],
            spans_disc=_spans(minute_of_day(e["start_dt"], cls), minute_of_day(e["last_dt"], cls)),
        )
        fresh = _default_event_state()
        # Only ACTIVE re-arms inside the same window, and only once the deviation has RESET under the
        # trigger. The exit tick itself counts: the fade leaves at <= norm_thr, already under
        # dev_thr, so nothing changes for it. The ride leaves at >= ext_thr, far above the trigger,
        # and must wait - without this it re-opened on the very tick it had just closed, which on the
        # test slice turned 25k real rides into 131k one-tick repeats.
        fresh["rearm_ok"] = IS_PASSIVE[cls] or (
            is_finite_num(e["last_dev_abs"]) and float(e["last_dev_abs"]) < dev_thr)
        events_by_ticker_cls[(ticker, cls)] = fresh

    def on_new_day(ticker):
        for _c in TRACKS:
            _end_run(ticker, _c)
        for _c in TRACKS:
            e_day = get_event(ticker, _c)
            if not e_day["active"]:
                continue
            start_t = ((e_day["start_dt"].hour, e_day["start_dt"].minute)
                        if isinstance(e_day["start_dt"], datetime) else None)
            # An event BORN IN THE EVENING is mid-flight across midnight by design - PRE and BLUE
            # wrap in both variants - so the day roll must not touch it; the grace loop in
            # process_chunk closes it at its own window instead. Anything born in the morning cannot
            # outlive its own day.
            if not (start_t is not None and start_t >= PRE_FROM):
                finalize_event(ticker, _c)

    # ---------------- main streaming pass ----------------
    cur_ticker = None
    cur_day = None

    def flush_ticker(ticker):
        for _c in TRACKS:
            _end_run(ticker, _c)
        for cls in TRACKS:
            if get_event(ticker, cls)["active"]:
                finalize_event(ticker, cls)
        for cls in TRACKS:
            events_by_ticker_cls.pop((ticker, cls), None)

    prev_state = {"tk": None, "dt": None}   # previous valid tick of the SAME ticker

    def process_chunk(chunk: "pd.DataFrame"):
        nonlocal cur_ticker, cur_day

        req = {"ticker", "date", "dt", "dev_sig"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Input must contain columns: {sorted(req)}")

        def col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_dt_ts = pd.to_datetime(col("dt"), errors="coerce", utc=True)
        s_dev = pd.to_numeric(col("dev_sig"), errors="coerce")

        dt_ok = s_dt_ts.notna()
        dev_ok = np.isfinite(s_dev.to_numpy(dtype="float64", copy=False))
        mask = (dt_ok.to_numpy(copy=False) & dev_ok)

        if mask.any():
            dt2 = s_dt_ts[mask]
            h = dt2.dt.hour.to_numpy(dtype="int16", copy=False)
            m = dt2.dt.minute.to_numpy(dtype="int16", copy=False)

            def _in_win(h, m, a, b):
                ah, am = a; bh, bm = b
                return ((h > ah) | ((h == ah) & (m >= am))) & ((h < bh) | ((h == bh) & (m <= bm)))

            ig = np.zeros_like(h, dtype=bool)
            for a, b in IGNORE_WINDOWS:
                ig |= _in_win(h, m, a, b)
            idx_mask = np.flatnonzero(mask)
            mask[idx_mask] = ~ig

        if not mask.any():
            return

        # .tolist() on each of these is what the row loop below actually wants: indexing a numpy
        # array boxes a fresh numpy scalar per read, and np.float64 arithmetic is several times
        # slower than plain float. The VALUES are unchanged - tolist() is an exact conversion for
        # float64/int16/object arrays, NaN included.
        dt_py = list(s_dt_ts[mask].dt.to_pydatetime())
        tk_arr = col("ticker")[mask].to_numpy(copy=False).tolist()
        ds_arr = col("date")[mask].to_numpy(copy=False).tolist()
        dev_arr = s_dev[mask].to_numpy(dtype="float64", copy=False).tolist()

        s_bench_name = col("bench")
        s_corr = col("corr"); s_beta = col("beta"); s_sigma = col("sigma")
        bench_name_arr = s_bench_name[mask].to_numpy(copy=False) if "bench" in chunk.columns else None
        corr_arr = s_corr[mask].to_numpy(copy=False) if "corr" in chunk.columns else None
        beta_arr = s_beta[mask].to_numpy(copy=False) if "beta" in chunk.columns else None
        sigma_arr = s_sigma[mask].to_numpy(copy=False) if "sigma" in chunk.columns else None

        s_bench_num = (pd.to_numeric(col(BENCH_NUM_FIELD), errors="coerce")
                       if BENCH_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))
        s_stock_pct = (pd.to_numeric(col(STOCK_NUM_FIELD), errors="coerce")
                       if STOCK_NUM_FIELD in chunk.columns else pd.Series(np.nan, index=chunk.index))
        bench_num_arr = s_bench_num[mask].to_numpy(dtype="float64", copy=False).tolist()
        stock_arr = s_stock_pct[mask].to_numpy(dtype="float64", copy=False).tolist()

        dt2 = s_dt_ts[mask]
        h_arr = dt2.dt.hour.to_numpy(dtype="int16", copy=False).tolist()
        m_arr = dt2.dt.minute.to_numpy(dtype="int16", copy=False).tolist()

        n = len(dev_arr)
        for i in range(n):
            tk = tk_arr[i]
            ds = ds_arr[i]
            dt_now = dt_py[i]
            dev_now = dev_arr[i]
            stock_pct = stock_arr[i]
            bench_num = bench_num_arr[i]
            t = (h_arr[i], m_arr[i])   # already python ints - see the .tolist() note above

            gap_prev = None
            if prev_state["tk"] == tk and prev_state["dt"] is not None:
                gap_prev = (dt_now - prev_state["dt"]).total_seconds() / 60.0
            prev_state["tk"], prev_state["dt"] = tk, dt_now

            if cur_ticker is not None and tk != cur_ticker:
                flush_ticker(cur_ticker)
                cur_ticker, cur_day = tk, ds
            if cur_ticker is None:
                cur_ticker, cur_day = tk, ds

            # setdefault builds its default on EVERY call, so the shipped line allocated a fresh
            # four-key dict per row - 1.8M throwaway dicts per row group - and used it only on the
            # first row of each ticker.
            st = static.get(tk)
            if st is None:
                st = static[tk] = {"bench": None, "corr": None, "beta": None, "sigma": None}
            if st["bench"] is None and bench_name_arr is not None:
                bn = bench_name_arr[i]
                if pd.notna(bn) and str(bn).strip():
                    st["bench"] = str(bn)
            if st["corr"] is None and corr_arr is not None and beta_arr is not None and sigma_arr is not None:
                c, b, s = corr_arr[i], beta_arr[i], sigma_arr[i]
                if pd.notna(c) and pd.notna(b) and pd.notna(s):
                    st["corr"], st["beta"], st["sigma"] = c, b, s

            if ds != cur_day:
                cur_day = ds
                on_new_day(tk)

            # One grace bound per track (GRACE_TO), replacing v13's PRE-only 09:40 plus a separate
            # `t > close` sweep for the day classes: every ARB class is overnight now, and three of
            # the five wrap midnight, where a bare wall-time comparison cannot say "past the close".
            for _c in TRACKS:
                e_g = get_event(tk, _c)
                if e_g["active"] and isinstance(e_g["start_dt"], datetime):
                    _g = GRACE_TO[_c]
                    _g_start_t = (e_g["start_dt"].hour, e_g["start_dt"].minute)
                    _close_dt = e_g["start_dt"].replace(hour=_g[0], minute=_g[1], second=0, microsecond=0)
                    if _g_start_t > _g:
                        _close_dt = _close_dt + timedelta(days=1)
                    if dt_now > _close_dt:
                        finalize_event(tk, _c)

            for cls in TRACKS:
                e = get_event(tk, cls)
                if not e["active"]:
                    if not e["rearm_ok"]:
                        wf2, wt2 = CLASS_WINDOWS[cls]
                        # Two ways the gate opens: the deviation resets under the trigger, or this
                        # track's window is simply over - the gate guards one session, and the state
                        # dict outlives it (the file is sorted by ticker, so one ticker's nights are
                        # streamed back to back).
                        if abs(float(dev_now)) < dev_thr or not (
                                in_range_wrap(t, wf2, wt2) if WRAP[cls] else in_range(t, wf2, wt2)):
                            e["rearm_ok"] = True
                    bf, bt = BIRTH_WINDOWS[cls]
                    born_inside = in_range_wrap(t, bf, bt) if WRAP[cls] else in_range(t, bf, bt)
                    # PASSIVE takes the FIRST birth of its session and nothing else; ACTIVE re-arms
                    # as often as the night offers.
                    if born_inside and IS_PASSIVE[cls] and passive_fired.get((tk, cls)) == session_key(dt_now, t):
                        born_inside = False
                    if born_inside and not e["rearm_ok"]:
                        born_inside = False
                    if born_inside and (not in_disc_guard(t)) and abs(float(dev_now)) >= dev_thr:
                        sg = 1 if float(dev_now) >= 0 else -1
                        r = runs.get((tk, cls))
                        if r is not None and r[1] == sg:
                            r[0] += 1
                        else:
                            _end_run(tk, cls)      # the other sign cut the previous run short
                            r = runs[(tk, cls)] = [1, sg, gap_prev]   # gap_prev = silence before the FIRST tick
                        if r[0] >= MIN_HOLD:
                            start_event(tk, cls, dt_now, ds, dev_now, stock_pct, bench_num, birth_gap_min=r[2])
                            if IS_PASSIVE[cls]:
                                passive_fired[(tk, cls)] = session_key(dt_now, t)
                            # start_event REBINDS the dict, so the `e` fetched above is now stale -
                            # this is the one branch that has to look it up again.
                            e = get_event(tk, cls)
                            runs.pop((tk, cls), None)
                    else:
                        _end_run(tk, cls)
                if e["active"]:
                    process_tick(tk, cls, dt_now, dev_now, stock_pct, bench_num, t, _e=e)

    t0 = time.time()
    total_rows = 0
    wanted = ["ticker", "date", "dt", "dev_sig", "bench", "corr", "beta", "sigma", STOCK_NUM_FIELD, BENCH_NUM_FIELD]
    cols = [c for c in wanted if c in pf.schema.names]
    for rg in range(pf.num_row_groups):
        chunk = pf.read_row_group(rg, columns=cols).to_pandas()
        process_chunk(chunk)
        total_rows += len(chunk)
        if (rg + 1) % parquet_batch_log_every == 0:
            print(f"[rg {rg+1}/{pf.num_row_groups}] rows={total_rows:,} elapsed={time.time()-t0:,.1f}s")
        del chunk
        if (rg + 1) % max(10, parquet_batch_log_every * 2) == 0:
            gc.collect()

    if cur_ticker is not None:
        flush_ticker(cur_ticker)

    # ---------------- all-history events -> OPG-style outputs ----------------
    t_post = time.time()
    ev = pd.DataFrame(all_events, columns=list(EV_COLS))
    all_events.clear()
    ev["cls"] = ev["cls"].str.upper()
    ev["pnl"] = pd.to_numeric(ev["pnl_pct"], errors="coerce")
    ev["entry_dev"] = ev["start_dev"].astype(float)
    ev["start_min"] = pd.to_numeric(ev["start_minute_idx"], errors="coerce")
    summary, lam_by_ticker = build_summary(ev, static, ARB_GAMMA_CFG, min_events_per_ticker)
    Path(ratings_path).parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(ratings_path, index=False)

    def _r(x, nd=4):
        return None if x is None or (isinstance(x, float) and not math.isfinite(x)) else round(float(x), nd)

    def _i(x):
        return None if x is None or (isinstance(x, float) and not math.isfinite(x)) else int(x)

    # the trade log keeps the newest MAX_WINDOW sessions only (the Scout file stays ~500k trades)
    log_dates = set(recent_dates)
    ratings_by_tk, pre_by_tk = {}, {}
    for r in summary.itertuples(index=False):
        ratings_by_tk.setdefault(r.ticker, []).append({
            "cls": r.cls, "sign": r.sign, "total": int(r.total), "wins": int(r.wins),
            "win_rate": _r(r.win_rate), "win_rate_lb": _r(r.win_rate_lb),
            "losses": int(r.losses), "pnl_n": int(r.pnl_n),
            # The money answer, as asked for: the SUM of the move, not its average.
            "sum_pnl": _r(r.sum_pnl, 2),
            "mean_pnl": _r(r.mean_pnl), "pnl_lb": _r(r.pnl_lb),
            "gamma": _r(r.gamma), "gamma_n": _i(r.gamma_n), "gamma_rate": _r(r.gamma_rate),
            "alpha": _r(r.alpha), "alpha_n": _i(r.alpha_n), "delta": _r(r.delta),
        })
        if r.cls == "PRE_ACT":
            pre_by_tk[(r.ticker, r.sign)] = r

    log = ev[ev["date"].isin(log_dates)]
    trades_by_tk = {}
    for tk, g in log.groupby("ticker", sort=False):
        trades_by_tk[tk] = [{
            "cls": t.cls.lower(), "sign": t.sign, "date": t.date,
            "start_dev": _r(t.start_dev), "start_gap_pct": _r(t.start_gap_pct), "status": t.status,
            "pnl_pct": _r(t.pnl_pct), "pnl_usd": None if t.pnl_pct is None or not math.isfinite(float(t.pnl_pct))
                                                   else round(position_usd * float(t.pnl_pct) / 100.0, 2),
            "start_minute_idx": _i(t.start_minute_idx), "peak_minute_idx": _i(t.peak_minute_idx),
            "peak_dev_abs": _r(t.peak_dev_abs), "end_dev_abs": _r(t.end_dev_abs),
            "end_gap_pct": _r(t.end_gap_pct), "exit_minute_idx": _i(t.exit_minute_idx),
            "birth_gap_min": None if t.birth_gap_min is None or not math.isfinite(float(t.birth_gap_min))
                             else round(float(t.birth_gap_min), 1),
            "spans_disc": bool(t.spans_disc), **({"alt": True} if t.is_alt else {}),
        } for t in g.sort_values(["date", "cls"], kind="stable").itertuples(index=False)]

    def _static_of(tk, field):
        v = (static.get(tk) or {}).get(field)
        if v is None:
            return None
        try:
            if isinstance(v, float) and not math.isfinite(v):
                return None
        except Exception:
            pass
        return str(v) if field == "bench" else round(float(v), 4)

    rows = []
    for tk in sorted(ratings_by_tk):
        pp, pn = pre_by_tk.get((tk, "pos")), pre_by_tk.get((tk, "neg"))
        rows.append({
            "ticker": tk,
            "alpha_pos": _r(pp.alpha) if pp is not None else None, "alpha_neg": _r(pn.alpha) if pn is not None else None,
            "delta_pos": _r(pp.delta) if pp is not None else None, "delta_neg": _r(pn.delta) if pn is not None else None,
            "bench": _static_of(tk, "bench"), "corr": _static_of(tk, "corr"),
            "beta": _static_of(tk, "beta"), "sigma": _static_of(tk, "sigma"),
            "lambda": _r(lam_by_ticker.get(tk)) if len(lam_by_ticker) else None,
            "ratings": ratings_by_tk[tk],
            "trades": trades_by_tk.get(tk, []),
        })

    payload = {
        "meta": {
            "version": "arbride-rolling-perf-v4-bands",
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "most_recent_session": all_dates_sorted[-1] if all_dates_sorted else None,
            "sessions_seen": n_sessions,
            "windows_days": list(WINDOWS),
            # the class set itself, so a consumer can tell a v4 file from a v3 one without reading a
            # date: v3 published PRE/OPEN/INTRA/POST, v4 publishes the overnight five.
            "classes": [c.upper() for c in CLASSES],
            "class_windows": {c.upper(): "births %02d:%02d-%02d:%02d, held to %02d:%02d"
                              % (BIRTH_WINDOWS[c][0] + BIRTH_WINDOWS[c][1] + CLASS_WINDOWS[c][1])
                              for c in CLASSES},
            "recent_dates": recent_dates,   # oldest->newest, the sessions the trade log covers
            "dev_thr": dev_thr, "ext_thr": ext_thr,
            "min_hold_ticks": MIN_HOLD, "min_events_per_ticker": min_events_per_ticker,
            "position_usd": position_usd,
            "outcome_rule": "ACTIVE win = the deviation EXTENDED (|dev_sig| >= ext_thr) before the 09:30 force-close; PASSIVE win = |dev_sig| at 09:30 was BIGGER than at entry. Binary, no hard/soft split",
            "variants": "ACTIVE = the event ends on the resolution and the track re-arms (several per night), force-closed at 09:30 if it never resolves; PASSIVE = one entry per session, out at 09:30 always. Entries: PRE/ARK to 09:00, BLUE to 03:59",
            "rating_window": "all sessions in final.parquet",
            "alpha_basis": "per (ticker, cls, sign): modal |start deviation| of every birth (0.1 bins, min 10)",
            "gamma_basis": "per (ticker, cls, sign): lowest start level >= 0.6 whose births have a positive 95% lower bound on mean pnl (min 10)",
            "delta_basis": "PRE only: median |start dev| of the first PRE birth of the night, last 5 nights per sign",
            "lambda_basis": "per ticker: RMS of the signed start deviation over every birth (min 2)",
            "pnl_basis": "real Stack% move, entry->the tick it came off (the resolution for a resolved ACTIVE event, the window close otherwise, 09:30 for PASSIVE), LONG on pos, SHORT on neg; sum_pnl is the SUM of it per cell",
        },
        "rows": rows,
    }
    with _open_text(output_path, "wt") as f:
        json.dump(payload, f, ensure_ascii=False)

    print(f"DONE rows_read={total_rows:,} events={len(ev):,} tickers_out={len(rows):,} "
          f"ratings_rows={len(summary):,} post={time.time() - t_post:,.0f}s -> {ratings_path} + {output_path}")


In [ ]:
# ------------------------------------------------------------------
# ARB•RIDE - the mirror of ARB•FADE: the same deviations, ridden instead of faded.
# ROLLING RECENCY PERFORMANCE (last 5/20/40/65 sessions) + P&L TRADE LOG, split by
# class x sign x start-deviation bucket. Feeds the "Scout" UI: pick ticker/class/
# sign/window, set a "start" threshold in %, sigma, gamma- or delta-relative units
# (gamma/delta come from best_params.jsonl.gz, joined by ticker) and get both a
# hit-rate and a 000-position P&L curve, entirely client-side against this file.
# ------------------------------------------------------------------

ROLLING_OUT_PATH = OUT_DIR / "arbride_rolling_perf.json.gz"
RATINGS_OUT_PATH = OUT_DIR / "arbride_ratings.csv"

devsig_rolling_perf_exporter(
    input_path=str(FINAL_PATH),
    output_path=str(ROLLING_OUT_PATH),
    ratings_path=str(RATINGS_OUT_PATH),

    windows_days=(5, 20, 40, 65),

    dev_thr=0.60,     # start: 0.6 sigma (dev_sig is already in sigma units)
    ext_thr=1.80,     # ride target: 3x the trigger, the mirror of FADE's 0.20 floor
    ignore_windows=(((3, 58), (4, 2)), ((7, 58), (8, 2))),   # dropped ALWAYS: no situation starts there
    min_hold_ticks=2,   # a signal must hold two consecutive ticks; 1 = the old single-tick birth

    start_bin_step=0.2,
    start_bin_max=3.0,

    min_events_per_ticker=10,
    min_events_per_cell=2,

    position_usd=1000.0,
    include_trade_log=True,
)

print("ARB•RIDE completed ->", RATINGS_OUT_PATH.name, ROLLING_OUT_PATH.name)
